# Full adaptive attacks against all defenses

The analysis is CPU-only: it does not load Qwen or Donut and is safe to run while GPU jobs are active. It displays the table in the notebook and saves report-ready 300-DPI PNG figures and CSV tables under `midterm/generated/`. If a future run is missing, the final cell explains which environment variable can point to it; rerun all cells after the run finishes.

This notebook compares two full 293-document, three-epsilon adaptive runs: uniform multi-view EOT/BPDA and frequency-aware hard-view EOT/BPDA. Every saved artifact remains a PNG; success is evaluated separately after JPEG, resize voting, selective multi-view voting, and cross-view/cross-prompt semantic consistency.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'midterm' / 'reporting.py').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from midterm.reporting import run_adaptive_defense_attack_report

## Adaptive robustness table and figure

Each denominator contains every clean-correct document in that dataset at that ε. `all_four` is the strict intersection: the same saved PNG reaches the target after every defense.

In [ ]:
uniform_result = run_adaptive_defense_attack_report(
    artifact_stem='uniform_adaptive_eot',
    title='Uniform multi-view EOT/BPDA',
)

## Frequency-aware hard-view attack

This method shifts sampling probability toward the highest-loss views and biases updates toward lower spatial frequencies that are more likely to survive JPEG and resizing.

In [ ]:
hardview_result = run_adaptive_defense_attack_report(
    run_candidates=[
        '/project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/outputs/jpeg_eot_attack/qwen_frequency_hardview_public_full_v1',
        'outputs/jpeg_eot_attack/qwen_frequency_hardview_public_full_v1',
    ],
    env_var='PP_HARDVIEW_ATTACK_RUN_DIR',
    artifact_stem='frequency_hardview_eot',
    title='Frequency-aware hard-view EOT/BPDA',
)

## Direct method comparison

Run this after both strict merges exist. The table compares the same datasets, ε values, and defenses under identical denominators.

In [ ]:
import pandas as pd
if uniform_result is not None and hardview_result is not None:
    uniform_summary = uniform_result[1].assign(method='uniform_eot')
    hardview_summary = hardview_result[1].assign(method='frequency_hardview')
    adaptive_comparison = pd.concat([uniform_summary, hardview_summary], ignore_index=True)
    display(adaptive_comparison.style.format({'adaptive_asr': '{:.1%}', 'median_lpips': '{:.3f}', 'median_ssim': '{:.3f}', 'median_runtime_seconds': '{:.1f}'}))
else:
    print('Run both adaptive experiments before building the direct comparison.')

## Interpretation checklist

Compare both adaptive ASRs with the earlier non-adaptive post-defense ASRs. A rise demonstrates adaptation; any remaining gap quantifies residual robustness. The hard-view method is better only if it improves defended ASR without an unacceptable LPIPS/SSIM or runtime cost. Avoid claiming that either method is guaranteed to defeat every input.